# Qwen3-1.7B QLoRA fine-tune (Colab T4)

Train the idea-provocation LoRA on Qwen3-1.7B in 4-bit (NF4) on a Colab T4. T4 has 16 GB VRAM and supports `bitsandbytes`, so the original QLoRA recipe works here even though it doesn't on the local Maxwell GPU.

**Before running, set runtime type to GPU:** Runtime → Change runtime type → Hardware accelerator: T4 GPU.

**Inputs you upload (one cell below):**
- `data/training_pairs.jsonl` from your local machine.

**Output (downloaded at the end):**
- `qwen3-1.7b-idea-lora.zip` — the LoRA adapter, ~40 MB.

Total wall time: 10-20 min.

In [ ]:
# 1. Install deps. T4 supports CUDA 12; use the standard wheels.
!pip -q install "transformers>=4.45" "peft>=0.13" "trl>=0.12" "datasets>=2.21" "bitsandbytes>=0.43" "accelerate>=0.34"

In [ ]:
# 2. Verify GPU.
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0))
print('vram MB:', torch.cuda.get_device_properties(0).total_memory // 1024**2)
props = torch.cuda.get_device_properties(0)
print(f'compute capability: {props.major}.{props.minor}')
assert props.major >= 7, 'bitsandbytes 4-bit needs compute capability >= 7.5; this GPU is too old.'

In [ ]:
# 3. Upload training_pairs.jsonl from your local machine.
from google.colab import files
from pathlib import Path
Path('data').mkdir(exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    if name.endswith('.jsonl'):
        Path('data/training_pairs.jsonl').write_bytes(content)
        print(f'wrote data/training_pairs.jsonl ({len(content)} bytes)')
        break

In [ ]:
# 4. Sanity-check the data.
import json
rows = [json.loads(l) for l in open('data/training_pairs.jsonl', encoding='utf-8')]
print(f'rows: {len(rows)}')
print(f'first input: {rows[0]["input"]!r}')
print(f'first output (chars): {len(rows[0]["output"])}')
print(f'sample output:\n{rows[0]["output"][:400]}...')

In [ ]:
# 5. Configure & train.
import json, os, torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL = 'Qwen/Qwen3-1.7B'
OUTPUT_DIR = '/content/output/qwen3-1.7b-idea-lora'
EPOCHS = 3
LR = 2e-4
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LEN = 1024
BATCH_SIZE = 4
GRAD_ACCUM = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

rows = [json.loads(l) for l in open('data/training_pairs.jsonl', encoding='utf-8')]
ds = Dataset.from_list([{'text': f"{r['input']}\n\n{r['output']}"} for r in rows])

sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=5,
    save_strategy='epoch',
    save_total_limit=1,
    bf16=False,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    max_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,
)

trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=ds, processing_class=tokenizer)
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('saved to', OUTPUT_DIR)

In [ ]:
# 6. Quick smoke test (use the already-trained model directly).
# `model` is the PEFT-wrapped model from cell 5 with trained adapter weights
# already in memory. Don't call PeftModel.from_pretrained on it -- that
# loads adapter-on-adapter and silently breaks (you sample from un-tuned base).
model.eval()

prompt = "What's a non-obvious way to think about Money?"
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
out = model.generate(
    **inputs,
    max_new_tokens=350,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    pad_token_id=tokenizer.pad_token_id,
)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# 7. Zip and download the adapter.
import shutil
shutil.make_archive('/content/qwen3-1.7b-idea-lora', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/qwen3-1.7b-idea-lora.zip')